# 20 — Knowledge Distillation (30B MoE Teacher → 8B Dense Student)

Transfers the trained Qwen3-Coder-30B-A3B MoE model's ARO expertise into a
Qwen3-8B dense model — 3x smaller, 3x faster, same architecture family.

## Why distill?

The pipeline produces a strong 30B MoE teacher (NB16-NB19) trained on ~1,800 samples.
But 30B MoE is 16 GB and slow. An 8B dense Qwen3 model is ~4.5 GB, 3x faster, and
reliable for tool calling — but fine-tuning it directly on 1,800 samples wastes
the 30B's generalization ability.

Distillation amplifies the data: the 30B generates ~10K high-quality ARO outputs
that the 8B student trains on. The student gets the teacher's reasoning, not just
the raw training data.

## Pipeline

```
1. Generate diverse prompts (actions x patterns x complexity)
2. Run each through the trained 30B MoE teacher
3. Validate with `aro check` — keep only passing outputs
4. Merge with original training samples
5. Fine-tune the Qwen3-8B student on the combined ~12K dataset
6. Evaluate student vs teacher on the test set
```

**Input:**
- Best 30B teacher model (from NB19/17/16)
- Knowledge base (`knowledge.json`) for prompt generation
- Original dataset (`05_dataset/train.jsonl`) for merging

**Output:**
- `data/06_distill/teacher_outputs.jsonl` — validated teacher generations
- `data/06_distill/student_train.jsonl` — merged training set for student
- `models/distill/student/fused/` — the distilled Qwen3-8B model

## Setup

In [ ]:
import sys, importlib, json, random, subprocess, tempfile, time, re
from pathlib import Path
from collections import Counter

_cfg_dir = Path('.').resolve()
if str(_cfg_dir) not in sys.path:
    sys.path.insert(0, str(_cfg_dir))
import config; importlib.reload(config); from config import *

DISTILL_DIR = DATA_ROOT / '06_distill'
DISTILL_DIR.mkdir(parents=True, exist_ok=True)
DISTILL_MODELS_DIR.mkdir(parents=True, exist_ok=True)

TEACHER_OUTPUTS  = DISTILL_DIR / 'teacher_outputs.jsonl'
STUDENT_TRAIN    = DISTILL_DIR / 'student_train.jsonl'
STUDENT_MLX_DIR  = DISTILL_DIR / 'mlx'
STUDENT_MLX_DIR.mkdir(parents=True, exist_ok=True)

# Generation settings
NUM_PROMPTS       = 5000     # diverse prompts to generate
TEACHER_TEMP      = 0.3     # temperature for teacher generation
TEACHER_MAX_TOKENS = 1024
MAX_RETRIES       = 2       # retries per prompt on aro check failure

print(f'Teacher model: {MODEL_ID}')
print(f'Student model: {STUDENT_MODEL_ID}')
print(f'Distill dir:   {DISTILL_DIR}')

## Step 1 — Load teacher model

Uses the best available model from the pipeline (DPO > iterative > fine-tune > warm-start).

In [ ]:
# Find the best teacher model (same logic as NB20)
def find_best_teacher():
    import re as _re
    def _highest_fused(parent, label):
        if not parent.exists(): return None
        rounds = sorted(
            [d for d in parent.glob('round_*/fused') if (d / 'config.json').exists()],
            key=lambda p: int(_re.search(r'round_(\d+)', str(p)).group(1))
        )
        if rounds:
            best = rounds[-1]
            print(f'Found {label}: {best}')
            return str(best)
        return None

    # Priority: DPO > iterative > finetune > warm-start > base
    dpo = MODELS_DIR / 'dpo' / 'fused'
    if dpo.exists() and (dpo / 'config.json').exists():
        print(f'Using DPO teacher: {dpo}')
        return str(dpo)
    r = _highest_fused(ITERATIVE_MODELS_DIR, 'iterative')
    if r: return r
    r = _highest_fused(FINETUNE_MODELS_DIR, 'finetune')
    if r: return r
    warm = resolve_warm_adapter()
    if warm:
        print(f'Using warm-start adapter as teacher')
        return None  # will load base + adapter
    print(f'WARNING: No fine-tuned teacher found — using base model')
    return None

teacher_path = find_best_teacher()

load_fn, generate_fn, make_sampler_fn = ensure_mlx_lm()

if teacher_path:
    print(f'Loading teacher from {teacher_path}...')
    teacher_model, teacher_tok = load_fn(teacher_path)
else:
    print(f'Loading base model with warm adapter...')
    teacher_model, teacher_tok, _, _, _ = load_model(with_adapter=True)

kb = load_knowledge()
SYSTEM_PROMPT = build_system_prompt(kb)
print(f'Teacher loaded. System prompt: {len(SYSTEM_PROMPT)} chars')

## Step 2 — Generate diverse prompts

Programmatically generates prompts from the knowledge base:
- Action combinations (53 actions × prepositions × patterns)
- Feature set templates (HTTP, events, file, socket, lifecycle)
- Multi-file application scenarios
- Q&A from proposal sections
- Error diagnosis / debugging scenarios
- Plugin usage scenarios

In [ ]:
random.seed(42)

actions = kb.get('actions', [])
action_verbs = [a['verbs'][0] for a in actions if a.get('verbs')]
action_roles = {a['verbs'][0]: a.get('role', 'own') for a in actions if a.get('verbs')}

# ── Prompt templates ──────────────────────────────────────────────────────

CODE_TEMPLATES = [
    # Single feature set
    'Write an ARO feature set that {task}.',
    'Create an ARO feature set named "{name}" that {task}.',
    'Write ARO code that {task}. Include proper error handling via Return.',
    # Application-Start
    'Write an ARO Application-Start that {startup_task}.',
    'Create a minimal ARO application that {startup_task} and keeps running with Keepalive.',
    # Multi-feature
    'Write an ARO application with two feature sets: one that {task1} and another that {task2}.',
    'Create an ARO HTTP API with feature sets for {crud_entity}: list, create, get by ID, and delete.',
    # Events
    'Write an ARO feature set that emits a {event_name} event, and a handler that processes it.',
    'Create an event-driven ARO application where {event_scenario}.',
    # Plugins
    'Write ARO code that uses a plugin qualifier to {qualifier_task}.',
    # For-each / iteration
    'Write an ARO feature set that iterates over a list of {items} and {loop_task} each one.',
    # Conditionals
    'Write an ARO feature set with a When guard that {condition_task}.',
    # File operations
    'Write an ARO feature set that reads a {file_type} file and {file_task}.',
]

QA_TEMPLATES = [
    'What is the {action} action in ARO and when would you use it?',
    'Explain the difference between {action1} and {action2} in ARO.',
    'How do you {concept} in ARO? Give an example.',
    'What are the rules for {topic} in ARO?',
    'When should you use {pattern} in ARO vs {alternative}?',
    'Explain how {feature} works in ARO with a code example.',
]

DEBUG_TEMPLATES = [
    'This ARO code has a bug. Find and fix it:\n\n```aro\n{broken_code}\n```',
    'Why does this ARO code fail `aro check`? Fix it:\n\n```aro\n{broken_code}\n```',
]

# ── Fill-in values ────────────────────────────────────────────────────────

tasks = [
    'retrieves a user by ID from a repository and returns it',
    'validates a request body and returns a 400 status on failure',
    'creates a new order with items from the request body',
    'sends a welcome email when a UserCreated event is received',
    'computes the total price from a list of items',
    'logs each HTTP request to the console with timestamp',
    'transforms a list of names to uppercase',
    'filters a list of products where price > 100',
    'reads a configuration file and starts the server',
    'publishes a notification event with the operation result',
    'fetches weather data from an external API and returns it',
    'stores a new record in the repository and emits a Created event',
    'extracts path parameters and query parameters from the request',
    'computes a hash of the password and stores the user',
    'merges two lists of items and removes duplicates',
    'watches a directory for file changes and logs them',
    'accepts a WebSocket connection and echoes messages',
    'reads a CSV file and transforms each row',
    'compares two dates and returns the later one',
    'renders a template with user data and returns HTML',
]

startup_tasks = [
    'starts an HTTP server with the OpenAPI contract',
    'reads a config file and starts a file watcher',
    'starts a TCP socket server on port 9000',
    'logs the application version and starts all services',
    'starts the HTTP server and a background file monitor',
]

entities = ['users', 'products', 'orders', 'sessions', 'articles', 'tasks', 'events']
events = ['UserCreated', 'OrderPlaced', 'PaymentReceived', 'FileChanged', 'SessionExpired']
concepts = [
    'emit custom events', 'use for-each loops', 'handle HTTP routes',
    'use Publish to share variables', 'create immutable transformations',
    'use repository observers', 'implement state machines with Accept',
    'configure runtime settings', 'use plugin qualifiers',
    'work with date/time ranges', 'split strings with regex',
    'use set operations on collections', 'render templates',
]
topics = [
    'variable immutability', 'feature set naming', 'Application-Start',
    'event-driven architecture', 'contract-first HTTP', 'string concatenation',
    'the Keepalive action', 'plugin qualifiers', 'store files',
]

broken_codes = [
    '(getUser: User API) {\n    Extract the <id> from <pathParameters>.\n    Retrieve the <user> from the <user-repository>.\n}',
    '(Application-Start: My App) {\n    Start the <http-server>.\n    Return an <OK: status> for the <startup>.\n}',
    '(handler: Events) {\n    Extract the <data> from the <event>.\n    Log <data> + " received" to the <console>.\n}',
    '(compute: Math) {\n    Compute the <total> = <price> * <quantity>.\n    Return the <total>.\n}',
]

# ── Generate prompts ──────────────────────────────────────────────────────

prompts = []

for _ in range(NUM_PROMPTS):
    roll = random.random()
    if roll < 0.55:  # 55% code generation
        tmpl = random.choice(CODE_TEMPLATES)
        prompt = tmpl.format(
            task=random.choice(tasks),
            task1=random.choice(tasks),
            task2=random.choice(tasks),
            name=f'{random.choice(["get", "create", "update", "delete", "list"])}{random.choice(entities).title()}',
            startup_task=random.choice(startup_tasks),
            crud_entity=random.choice(entities),
            event_name=random.choice(events),
            event_scenario=f'a {random.choice(events)} triggers a notification',
            qualifier_task=f'sort a list of {random.choice(entities)}',
            items=random.choice(entities),
            loop_task=random.choice(['logs', 'transforms', 'validates', 'stores']),
            condition_task=f'checks if the {random.choice(entities)} list is empty',
            file_type=random.choice(['JSON', 'YAML', 'CSV', 'text']),
            file_task=random.choice(['logs its contents', 'transforms the data', 'stores each record']),
        )
        prompts.append(('code_generation', prompt))
    elif roll < 0.80:  # 25% Q&A
        tmpl = random.choice(QA_TEMPLATES)
        prompt = tmpl.format(
            action=random.choice(action_verbs),
            action1=random.choice(action_verbs),
            action2=random.choice(action_verbs),
            concept=random.choice(concepts),
            topic=random.choice(topics),
            pattern=random.choice(['Publish', 'Emit', 'Store', 'For-each']),
            alternative=random.choice(['Return', 'Log', 'Compute', 'Extract']),
            feature=random.choice(concepts),
        )
        prompts.append(('syntax_qa', prompt))
    elif roll < 0.95:  # 15% debugging
        tmpl = random.choice(DEBUG_TEMPLATES)
        prompt = tmpl.format(broken_code=random.choice(broken_codes))
        prompts.append(('debugging', prompt))
    else:  # 5% explanation
        prompt = f'Explain this ARO code:\n\n```aro\n{random.choice(broken_codes)}\n```'
        prompts.append(('code_explanation', prompt))

# Deduplicate
seen = set()
unique_prompts = []
for task_type, p in prompts:
    key = p[:200]
    if key not in seen:
        seen.add(key)
        unique_prompts.append((task_type, p))

random.shuffle(unique_prompts)
print(f'Generated {len(unique_prompts)} unique prompts')
print('Distribution:', dict(Counter(t for t, _ in unique_prompts)))

## Step 3 — Run teacher and validate

For each prompt, the teacher generates a response. Code outputs are validated
with `aro check`. Failed outputs are retried up to `MAX_RETRIES` times at
higher temperature. Only validated outputs are kept.

In [ ]:
def generate_teacher(prompt, temperature=TEACHER_TEMP, max_tokens=TEACHER_MAX_TOKENS):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': prompt},
    ]
    text = teacher_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    tokens = teacher_tok.encode(text)
    reply = generate_fn(
        teacher_model, teacher_tok,
        prompt=tokens,
        max_tokens=max_tokens,
        sampler=make_sampler_fn(temp=temperature, top_p=0.9),
        verbose=False,
    )
    return reply.strip()


def extract_aro_blocks(text):
    return [b.strip() for b in re.findall(r'```aro\n(.*?)```', text, re.DOTALL) if b.strip()]


def aro_check(code):
    try:
        with tempfile.TemporaryDirectory() as tmp:
            (Path(tmp) / 'main.aro').write_text(code)
            r = subprocess.run(['aro', 'check', tmp], capture_output=True, text=True, timeout=10)
            return r.returncode == 0
    except Exception:
        return False


# ── Generate and validate ─────────────────────────────────────────────────
teacher_samples = []
passed, failed, qa_kept = 0, 0, 0

# Resume from existing outputs if re-running
existing = set()
if TEACHER_OUTPUTS.exists():
    with open(TEACHER_OUTPUTS) as f:
        for line in f:
            if line.strip():
                rec = json.loads(line)
                existing.add(rec.get('prompt', '')[:200])
                teacher_samples.append(rec)
    print(f'Resuming: {len(existing)} samples already generated')

t0 = time.time()

for i, (task_type, prompt) in enumerate(unique_prompts):
    if prompt[:200] in existing:
        continue

    if (i + 1) % 100 == 0:
        elapsed = time.time() - t0
        rate = (passed + qa_kept) / max(1, elapsed) * 3600
        print(f'  [{i+1}/{len(unique_prompts)}] passed={passed} qa={qa_kept} failed={failed} ({rate:.0f}/hr)')

    # Generate teacher output
    for attempt in range(MAX_RETRIES + 1):
        temp = TEACHER_TEMP + attempt * 0.1  # increase temp on retries
        output = generate_teacher(prompt, temperature=temp)

        if task_type in ('syntax_qa', 'code_explanation'):
            # Q&A: just check it's not empty
            if len(output) > 30:
                teacher_samples.append({
                    'task_type': task_type,
                    'prompt': prompt,
                    'output': output,
                    'source': 'distill_teacher',
                    'valid': True,
                })
                qa_kept += 1
                break
        else:
            # Code: validate with aro check
            blocks = extract_aro_blocks(output)
            if blocks:
                code = '\n\n'.join(blocks)
                if aro_check(code):
                    teacher_samples.append({
                        'task_type': task_type,
                        'prompt': prompt,
                        'output': output,
                        'source': 'distill_teacher',
                        'valid': True,
                    })
                    passed += 1
                    break
        if attempt == MAX_RETRIES:
            failed += 1

    # Save incrementally every 50 samples
    if len(teacher_samples) % 50 == 0:
        with open(TEACHER_OUTPUTS, 'w') as f:
            for s in teacher_samples:
                f.write(json.dumps(s) + '\n')

# Final save
with open(TEACHER_OUTPUTS, 'w') as f:
    for s in teacher_samples:
        f.write(json.dumps(s) + '\n')

elapsed = time.time() - t0
print(f'\nDone in {elapsed/60:.1f} min')
print(f'  Teacher outputs: {len(teacher_samples)}')
print(f'  Code passed:     {passed}')
print(f'  Q&A kept:        {qa_kept}')
print(f'  Failed:          {failed}')
print(f'  Pass rate:       {(passed + qa_kept) / max(1, passed + qa_kept + failed):.0%}')

## Step 4 — Merge with original training data

Combines teacher outputs with the original 1,800 training samples from NB15.
The original samples are high-quality human-curated data — they anchor the
student to the ground truth while the teacher outputs expand coverage.

In [ ]:
# Load original training data
original_train = []
orig_path = DATA_ROOT / '05_dataset' / 'train.jsonl'
if orig_path.exists():
    with open(orig_path) as f:
        for line in f:
            if line.strip():
                original_train.append(json.loads(line))
    print(f'Original training samples: {len(original_train)}')
else:
    print(f'WARNING: {orig_path} not found — run NB15 first')

# Convert teacher outputs to ChatML format
teacher_chatml = []
for s in teacher_samples:
    teacher_chatml.append({
        'messages': [
            {'role': 'system',    'content': SYSTEM_PROMPT},
            {'role': 'user',      'content': s['prompt']},
            {'role': 'assistant', 'content': s['output']},
        ],
        'task_type': s['task_type'],
        'source': 'distill_teacher',
    })

# Merge: original first (higher weight), then teacher
merged = original_train + teacher_chatml

# Deduplicate by user message prefix
seen = set()
deduped = []
for s in merged:
    key = s['messages'][1]['content'][:300]
    if key not in seen:
        seen.add(key)
        deduped.append(s)

random.seed(42)
random.shuffle(deduped)

# Split: 95% train, 5% valid
n = len(deduped)
split = int(n * 0.95)
train = deduped[:split]
valid = deduped[split:]

# Save full format
with open(STUDENT_TRAIN, 'w') as f:
    for s in train:
        f.write(json.dumps(s) + '\n')

# Save MLX format (messages only)
with open(STUDENT_MLX_DIR / 'train.jsonl', 'w') as f:
    for s in train:
        f.write(json.dumps({'messages': s['messages']}) + '\n')
with open(STUDENT_MLX_DIR / 'valid.jsonl', 'w') as f:
    for s in valid:
        f.write(json.dumps({'messages': s['messages']}) + '\n')

print(f'\nMerged dataset:')
print(f'  Original:      {len(original_train)}')
print(f'  Teacher:       {len(teacher_chatml)}')
print(f'  After dedup:   {len(deduped)}')
print(f'  Train:         {len(train)}')
print(f'  Valid:         {len(valid)}')
print(f'\nTask distribution:')
dist = Counter(s.get('task_type', s.get('source', '?')) for s in train)
for k, v in sorted(dist.items(), key=lambda x: -x[1]):
    print(f'  {k:25s}: {v}')

## Step 5 — Fine-tune student model

LoRA fine-tune the 8B student on the merged dataset.
Same hyperparameters as NB16, adjusted for the larger dataset.

In [ ]:
# Free teacher model memory
del teacher_model
import gc; gc.collect()
print('Teacher model freed.')

STUDENT_ADAPTER = DISTILL_MODELS_DIR / 'student' / 'adapter'
STUDENT_FUSED   = DISTILL_MODELS_DIR / 'student' / 'fused'
STUDENT_ADAPTER.mkdir(parents=True, exist_ok=True)

num_iters = min(1200, len(train) * 2)  # ~2 epochs

cmd = [
    sys.executable, '-m', 'mlx_lm', 'lora',
    '--model',           STUDENT_MODEL_ID,
    '--data',            str(STUDENT_MLX_DIR),
    '--adapter-path',    str(STUDENT_ADAPTER),
    '--train',
    '--use-chat-template',
    '--batch-size',      '1',
    '--grad-accumulate', '16',
    '--lora-layers',     '8',
    '--lora-rank',       '16',
    '--learning-rate',   '2e-5',
    '--iters',           str(num_iters),
    '--steps-per-report', '50',
    '--steps-per-eval',  '100',
    '--save-every',      '200',
    '--max-seq-length',  '2048',
]

print(f'Training student ({STUDENT_MODEL_ID}) on {len(train)} samples for {num_iters} iterations...')
print(f'Command: {" ".join(cmd)}\n')

result = subprocess.run(cmd, text=True)
if result.returncode != 0:
    print(f'Training failed with exit code {result.returncode}')
else:
    print(f'\nTraining complete. Adapter: {STUDENT_ADAPTER}')

## Step 6 — Fuse student adapter

In [ ]:
if not (STUDENT_FUSED / 'config.json').exists():
    print(f'Fusing adapter into {STUDENT_FUSED}...')
    cmd = [
        sys.executable, '-m', 'mlx_lm', 'fuse',
        '--model',        STUDENT_MODEL_ID,
        '--adapter-path', str(STUDENT_ADAPTER),
        '--save-path',    str(STUDENT_FUSED),
    ]
    subprocess.run(cmd, check=True)
    print('Fused.')
else:
    print(f'Student fused model already exists: {STUDENT_FUSED}')

## Step 7 — Evaluate student vs teacher

Run the test set through both models and compare syntax pass rate + output quality.

In [ ]:
# Load test set
test_path = DATA_ROOT / '05_dataset' / 'test.jsonl'
test_samples = []
if test_path.exists():
    with open(test_path) as f:
        for line in f:
            if line.strip():
                test_samples.append(json.loads(line))
print(f'Test samples: {len(test_samples)}')

# Load student model
print(f'Loading student from {STUDENT_FUSED}...')
student_model, student_tok = load_fn(str(STUDENT_FUSED))
print('Student loaded.')


def evaluate_model(model, tokenizer, label):
    code_pass, code_fail, qa_ok, qa_thin = 0, 0, 0, 0
    for s in test_samples:
        prompt = s['messages'][1]['content']
        task = s.get('task_type', '')

        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user',   'content': prompt},
        ]
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        tokens = tokenizer.encode(text)
        reply = generate_fn(
            model, tokenizer,
            prompt=tokens,
            max_tokens=512,
            sampler=make_sampler_fn(temp=0.1),
            verbose=False,
        ).strip()

        if task in ('code_generation', 'debugging', 'fim'):
            blocks = extract_aro_blocks(reply)
            if blocks and aro_check('\n\n'.join(blocks)):
                code_pass += 1
            else:
                code_fail += 1
        else:
            text_only = re.sub(r'```.*?```', '', reply, flags=re.DOTALL).strip()
            if len(text_only) > 30:
                qa_ok += 1
            else:
                qa_thin += 1

    total_code = code_pass + code_fail
    total_qa = qa_ok + qa_thin
    print(f'\n{label}:')
    print(f'  Code:  {code_pass}/{total_code} pass ({100*code_pass/max(1,total_code):.0f}%)')
    print(f'  Q&A:   {qa_ok}/{total_qa} answered ({100*qa_ok/max(1,total_qa):.0f}%)')
    return {'code_pass': code_pass, 'code_total': total_code, 'qa_ok': qa_ok, 'qa_total': total_qa}


student_metrics = evaluate_model(student_model, student_tok, 'Student (8B distilled)')

# Optionally evaluate teacher too (requires reloading — skip if memory constrained)
# teacher_metrics = evaluate_model(teacher_model, teacher_tok, 'Teacher (30B)')

# Save metrics
metrics = {
    'student_model': STUDENT_MODEL_ID,
    'teacher_model': str(teacher_path or MODEL_ID),
    'teacher_outputs': len(teacher_samples),
    'student_train_samples': len(train),
    'student_metrics': student_metrics,
}
with open(DISTILL_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\nMetrics saved to {DISTILL_DIR / "metrics.json"}')

## Summary

In [ ]:
print('=' * 65)
print('notebook 20 — DISTILLATION SUMMARY')
print('=' * 65)
print(f'\nTeacher: {teacher_path or MODEL_ID}')
print(f'Student: {STUDENT_MODEL_ID}')
print(f'\nData amplification:')
print(f'  Original samples:   {len(original_train):>6,}')
print(f'  Teacher outputs:    {len(teacher_samples):>6,}')
print(f'  Merged (deduped):   {len(deduped):>6,}')
print(f'  Student train set:  {len(train):>6,}')
print(f'\nStudent model: {STUDENT_FUSED}')
if student_metrics:
    cp = student_metrics['code_pass']
    ct = student_metrics['code_total']
    print(f'  Syntax pass rate: {cp}/{ct} ({100*cp/max(1,ct):.0f}%)')
print(f'\nNext: run NB21 to package the student model for distribution.')
print(f'  The student model at {STUDENT_FUSED}')
print(f'  will be picked up as the best available model.')
print('=' * 65)